<a href="https://colab.research.google.com/github/jgschmitz/MongoDB-Demos/blob/master/mdbr_leaf_ir_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# mdbr-leaf-ir: Mini Retrieval Demo (Colab-ready)

This notebook shows a tiny **retrieval** demo using **[`MongoDB/mdbr-leaf-ir`](https://huggingface.co/MongoDB/mdbr-leaf-ir)**.
It supports both **symmetric** (same model for queries & docs) and **asymmetric** (different models) setups.

> **How to use**: Run top-to-bottom. If you're on Colab, it will install needed packages automatically.


In [ ]:
# If running on Colab or a fresh environment, uncomment the next cell to install dependencies.
# You can safely re-run this cell.
# %%capture
# !pip install -q --upgrade sentence-transformers faiss-cpu transformers accelerate torch --index-url https://download.pytorch.org/whl/cpu


## 1) Imports & device


In [ ]:
from __future__ import annotations

import os, math
from typing import List, Tuple

try:
    import torch
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
except Exception:
    device = 'cpu'

print(f'Running on device: {device}')


## 2) Configure models

- **Symmetric**: use `MongoDB/mdbr-leaf-ir` for both queries and documents.
- **Asymmetric**: keep `mdbr-leaf-ir` for **queries** and use a **larger document encoder** (e.g., `Snowflake/snowflake-arctic-embed-m-v1.5`).  
  This can help quality at slightly higher doc-embed cost.


In [ ]:
USE_ASYMMETRIC = False  # set True to try asymmetric retrieval

QUERY_MODEL_NAME = "MongoDB/mdbr-leaf-ir"
DOC_MODEL_NAME   = "MongoDB/mdbr-leaf-ir" if not USE_ASYMMETRIC else "Snowflake/snowflake-arctic-embed-m-v1.5"

print("Query encoder :", QUERY_MODEL_NAME)
print("Document encoder:", DOC_MODEL_NAME)


## 3) Load encoders


In [ ]:
from sentence_transformers import SentenceTransformer

query_model = SentenceTransformer(QUERY_MODEL_NAME, device=device)
doc_model   = query_model if DOC_MODEL_NAME == QUERY_MODEL_NAME else SentenceTransformer(DOC_MODEL_NAME, device=device)

# Small helper to normalize to unit length for cosine/IP search
import numpy as np
def _normalize(x: np.ndarray) -> np.ndarray:
    n = np.linalg.norm(x, axis=1, keepdims=True) + 1e-12
    return x / n


## 4) Toy corpus
Feel free to edit these documents to your own domain.


In [ ]:
corpus = [
    "Prior authorization requests require clinical documentation and payer rules validation.",
    "Members can schedule appointments online or via the call center.",
    "Metformin is a first-line therapy for type 2 diabetes.",
    "Claims can be appealed within 60 days of the initial decision.",
    "Temporal orchestrates workflows while MongoDB stores operational state.",
    "Hypertension guidelines recommend lifestyle changes and medication titration.",
    "Care managers track outreach attempts, assessments, and care plans.",
    "FHIR resources standardize healthcare data exchange across systems."
]

doc_emb = doc_model.encode(corpus, convert_to_numpy=True, normalize_embeddings=False, show_progress_bar=False)
doc_emb = _normalize(doc_emb)
dim = doc_emb.shape[1]
print(f"Embedded {len(corpus)} docs with dimension = {dim}")


## 5) Build a FAISS index (inner product = cosine on normalized vectors)


In [ ]:
import faiss

index = faiss.IndexFlatIP(dim)  # inner product
index.add(doc_emb.astype('float32'))
print("FAISS index size:", index.ntotal)


## 6) Simple search helper


In [ ]:
def search(query: str, k: int = 5) -> List[Tuple[float, str]]:
    q_emb = query_model.encode([query], convert_to_numpy=True, normalize_embeddings=False, show_progress_bar=False)
    q_emb = _normalize(q_emb).astype('float32')
    scores, idx = index.search(q_emb, k)
    results = [(float(scores[0][i]), corpus[int(idx[0][i])]) for i in range(len(idx[0]))]
    return results

def pretty_print(query: str, k: int = 5):
    print(f"\nQuery: {query}")
    for score, text in search(query, k):
        print(f"  score={score:.3f}  |  {text}")


## 7) Try a few queries


In [ ]:
pretty_print("appeal a denied claim", k=3)
pretty_print("type 2 diabetes medication", k=3)
pretty_print("who stores the operational state with temporal", k=3)


## 8) Your turn
Run this cell and type your own query when prompted.


In [ ]:
try:
    q = input("Enter a query: ").strip()
    if q:
        pretty_print(q, k=5)
    else:
        print("No query entered.")
except EOFError:
    # In some notebook environments input() may not be available.
    pretty_print("care plan tracking", k=5)


---

## (Optional) Asymmetric retrieval quick switch
Set `USE_ASYMMETRIC = True` in the config cell and re-run the notebook.  
This encodes **documents** with a larger model (e.g., Arctic Embed M) and **queries** with `mdbr-leaf-ir` for lower-latency interactive search.


## (Optional) Atlas Vector Search starter (manual placeholders)

> This section is **optional** and only runs if you provide a `MONGODB_URI`.  
> It shows how to persist your docs + embeddings to MongoDB Atlas and query with a `$vectorSearch` pipeline once you've created a compatible vector index.

1. Create a collection, e.g. `demo.corpus` with fields:
   - `_id: ObjectId`
   - `text: string`
   - `embedding: array<float>` (dimension must match your encoder)

2. Create an Atlas Vector Search index on `embedding` with the correct dimension.

3. Run the following cells to insert and query.


In [ ]:
# OPTIONAL: Uncomment to use Atlas. Requires: pip install pymongo
# from pymongo import MongoClient
# import os, json
#
# uri = os.environ.get("MONGODB_URI")  # <-- set this env var
# if not uri:
#     raise ValueError("Set MONGODB_URI env var to your Atlas connection string")
#
# client = MongoClient(uri)
# coll = client.demo.corpus
#
# # Upsert docs with embeddings
# docs = []
# for i, (text, emb) in enumerate(zip(corpus, doc_emb)):
#     docs.append({
#         "_id": i,
#         "text": text,
#         "embedding": emb.astype(float).tolist()
#     })
# coll.delete_many({})
# coll.insert_many(docs)
# print("Inserted", coll.count_documents({}), "docs.")
#
# # Vector search pipeline (MongoDB 7.0+)
# q = "appeal a denied claim"
# q_emb = _normalize(query_model.encode([q], convert_to_numpy=True)).astype(float).tolist()[0]
# pipeline = [
#     {
#       "$vectorSearch": {
#         "index": "vector_idx",
#         "path": "embedding",
#         "queryVector": q_emb,
#         "numCandidates": 100,
#         "limit": 5
#       }
#     },
#     { "$project": { "_id": 0, "text": 1, "score": { "$meta": "vectorSearchScore" } } }
# ]
# for doc in coll.aggregate(pipeline):
#     print(doc)
